# 172 — Razonamiento y cómputo en tiempo de inferencia

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Mayoría con p = 0.7

```text
k=3: C(3,2)·0.7²·0.3 + 0.7³ = 3·0.49·0.3 + 0.343 = 0.441 + 0.343 = 0.784
k=5: C(5,3)·0.7³·0.3² + C(5,4)·0.7⁴·0.3 + 0.7⁵
   = 10·0.343·0.09 + 5·0.2401·0.3 + 0.16807
   = 0.3087 + 0.36015 + 0.16807 ≈ 0.8369
```

De 3 a 5 muestras: +5.3 puntos (0.784 → 0.837), frente a +8.4 de 1 a 3.
Rendimientos decrecientes con coste lineal: cada punto extra de precisión
cuesta más muestras que el anterior.


In [ ]:
from math import comb
def maj(p, k):
    return sum(comb(k, i) * p**i * (1 - p)**(k - i) for i in range((k // 2) + 1, k + 1))
p = 0.7
maj3, maj5 = maj(p, 3), maj(p, 5)
print(f"k=1: {p}  k=3: {maj3:.4f}  k=5: {maj5:.4f}")
assert abs(maj3 - 0.784) < 1e-3 and abs(maj5 - 0.8369) < 1e-3


## Solución 2 — Votar con sesgo sistemático

a) P(mayoría de 5 correcta) con p = 0.45 = maj(0.45, 5) ≈ **0.4069**.

b) Empeoró: 0.4069 < 0.45. Cuando el error apunta consistentemente a la misma
respuesta, esa respuesta errónea es el modo y el voto la consolida.

c) Con errores repartidos entre 4 respuestas distintas, cada una acumula en
promedio 0.55/4 ≈ 0.1375 de masa, muy por debajo de 0.45: la respuesta correcta
sigue siendo el **modo** aunque p < 0.5, y el voto por pluralidad la selecciona
cada vez más a medida que crece k. La condición útil no es p > 0.5, sino "la
respuesta correcta es la más probable individualmente" (modo de la
distribución de respuestas).


In [ ]:
maj5_sesgo = maj(0.45, 5)
print(f"mayoría binaria con p=0.45: {maj5_sesgo:.4f} (< 0.45: votar empeora)")
assert maj5_sesgo < 0.45


## Solución 3 — Presupuesto adaptativo

b) Óptimo: FÁCIL 1 muestra (4 u.), IMPOSIBLE 1 muestra (4 u.) — o cero si el
sistema permite abstenerse—, y el resto a la MEDIA: 52 u. → 13 muestras, pero
la media satura en 80 % con 1+10 muestras (5 incrementos de +5 desde 55 %:
55→60→65→70→75→80). Basta con 11 muestras (44 u.) para saturar; el sobrante no
compra nada. Precisión media ≈ (0.95 + 0.80 + 0.0) / 3 ≈ **0.583**.

a/c) Cualquier asignación que gaste en la fácil o en la imposible pierde:
exactamente el resultado de Snell et al. — el cómputo de inferencia rinde en la
franja de dificultad intermedia; asignarlo *adaptativamente* supera a
repartirlo uniforme (y a comprar un modelo mayor con el mismo gasto).


## Solución 4 — Subir la claim a current

Evidencia mínima exigible:

1. **Comparación a igualdad de cómputo**: el modelo grande también con 32
   muestras (o el pequeño contado en FLOPs totales) — si no, el resultado puede
   ser solo "más cómputo gana", no "self-consistency gana".
2. **Replicación externa** con código/semillas públicas y más de un benchmark
   (GSM8K está parcialmente contaminado en muchos corpus de entrenamiento).
3. **Generalización fuera del dominio verificable**: resultados en tareas sin
   respuesta numérica extraíble, o la claim debe restringirse explícitamente a
   dominios tipo GSM8K.


In [ ]:
result = run_lab("frontier", seed=172)
evidencia_requerida = [
    "comparación a igualdad de FLOPs totales con el modelo grande",
    "replicación externa en más de un benchmark no contaminado",
    "alcance declarado: solo dominios con respuesta verificable",
]
for e in evidencia_requerida:
    print("-", e)
